# Detecting Fake Websites with Machine Learning
### Group No: 15
#### F21DL Data Mining and Machine Learning, Group Coursework
---

| Name | Student number | Contribution |
|---|---|---|
| Arafa Mehar | H | R1, R2 |
| Syed Umraaz | H | R1, R2 |
| Sakib Dalal | H | R1, R3 |
| Eric | H | R1, R4 |
| Rishab Dhinesh | H | R1, R4 |

---

- Datasets:
    - PhiUSIIL Phishing URL Dataset, UCI Machine Learning Repository (ID 967), CC BY 4.0
    - Image CNN Dataset

---

## Notebook Index
All of the code for requirements R1 to R4 is in this notebook.
1. The questions and the hypotheses (R1)
2. Loading and cleaning the data (R2)
3. Looking at the data with graphs (R2)
4. Choosing the features (R2)
5. Clustering (R2)
6. The classical models (R3)
7. The decision tree (R3)
8. The neural networks (R4)
9. The final comparison and the conclusion
   
---


## Imports and settings

In [1]:
import os
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")
os.environ["TRANSFORMERS_VERBOSITY"] = "error"

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 180)
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)

TEST_SIZE = 0.30
CLASS_NAMES = ["Phishing", "Legitimate"]

SIMILARITY_FEATURES = ["URLSimilarityIndex", "URLTitleMatchScore", "DomainTitleMatchScore"]

Matplotlib is building the font cache; this may take a moment.


---
# Step 01: *Questions and the hypotheses (R1)*
---

---
# Step 02: *Loading and cleaning the data (R2)*

The dataset is one CSV file. The file is in the `Datasets` folder, which is one level above this notebook, so the path starts with `..`.

---

In [2]:
DATASET_FILE = "PhiUSIIL_Phishing_URL_Dataset.csv"
DATA_PATH = os.path.join("..", "Datasets", DATASET_FILE)

print("File found:", os.path.exists(DATA_PATH))

File found: True


In [3]:
df = pd.read_csv(DATA_PATH)

print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

Rows: 235795
Columns: 56


In [4]:
df.head()

,FILENAME,URL,URLLength,Domain,DomainLength,IsDomainIP,TLD,URLSimilarityIndex,CharContinuationRate,TLDLegitimateProb,URLCharProb,TLDLength,NoOfSubDomain,HasObfuscation,NoOfObfuscatedChar,ObfuscationRatio,NoOfLettersInURL,LetterRatioInURL,NoOfDegitsInURL,DegitRatioInURL,NoOfEqualsInURL,NoOfQMarkInURL,NoOfAmpersandInURL,NoOfOtherSpecialCharsInURL,SpacialCharRatioInURL,IsHTTPS,LineOfCode,LargestLineLength,HasTitle,Title,DomainTitleMatchScore,URLTitleMatchScore,HasFavicon,Robots,IsResponsive,NoOfURLRedirect,NoOfSelfRedirect,HasDescription,NoOfPopup,NoOfiFrame,HasExternalFormSubmit,HasSocialNet,HasSubmitButton,HasHiddenFields,HasPasswordField,Bank,Pay,Crypto,HasCopyrightInfo,NoOfImage,NoOfCSS,NoOfJS,NoOfSelfRef,NoOfEmptyRef,NoOfExternalRef,label
0,521848.txt,https://www.southbankmosaics.com,31,www.southbankmosaics.com,24,0,com,100.0,1.000000,0.522907,0.061933,3,1,0,0,0.0,18,0.581,0,0.0,0,0,0,1,0.032,1,558,9381,1,à¸‚à¹ˆà¸²à¸§à¸ªà¸” à¸‚à¹ˆà¸²à¸§à¸§à¸±à¸™à¸™à¸µ...,0.000000,0.000000,0,1,1,0,0,0,0,1,0,0,1,1,0,1,0,0,1,34,20,28,119,0,124,1
1,31372.txt,https://www.uni-mainz.de,23,www.uni-mainz.de,16,0,de,100.0,0.666667,0.032650,0.050207,2,1,0,0,0.0,9,0.391,0,0.0,0,0,0,2,0.087,1,618,9381,1,johannes gutenberg-universitÃ¤t mainz,55.555556,55.555556,1,1,0,0,0,0,0,0,0,1,1,0,0,0,0,0,1,50,9,8,39,0,217,1
2,597387.txt,https://www.voicefmradio.co.uk,29,www.voicefmradio.co.uk,22,0,uk,100.0,0.866667,0.028555,0.064129,2,2,0,0,0.0,15,0.517,0,0.0,0,0,0,2,0.069,1,467,682,1,voice fm southampton,46.666667,46.666667,0,1,1,0,0,1,0,0,0,0,1,1,0,0,0,0,1,10,2,7,42,2,5,1
3,554095.txt,https://www.sfnmjournal.com,26,www.sfnmjournal.com,19,0,com,100.0,1.000000,0.522907,0.057606,3,1,0,0,0.0,13,0.500,0,0.0,0,0,0,1,0.038,1,6356,26824,1,home page: seminars in fetal and neonatal medi...,0.000000,0.000000,0,1,1,0,0,0,1,12,0,1,1,1,0,0,1,1,1,3,27,15,22,1,31,1
4,151578.txt,https://www.rewildingargentina.org,33,www.rewildingargentina.org,26,0,org,100.0,1.000000,0.079963,0.059441,3,1,0,0,0.0,20,0.606,0,0.0,0,0,0,1,0.030,1,6089,28404,1,fundaciÃ³n rewilding argentina,100.000000,100.000000,0,1,1,1,1,1,0,2,0,1,1,1,0,1,1,0,1,244,15,34,72,1,85,1


The columns we care about are these.

- `URL` is the address of the website. It is text.
- `label` is the answer. It is 1 for a real website and 0 for a phishing website.
- The rest are numbers about the URL and about the page, like `URLLength`, `IsHTTPS`, `LineOfCode`, `NoOfImage` and `NoOfExternalRef`.

---
# Step 03: *Looking at the data with graphs (R2)*
---

---
# Step 04: *Choosing the features (R2)*
---

---
# Step 05: *Clustering (R2)*
---

---
# Step 06: *Classical Models (R3)*
---

---
# Step 07: *Decision Tree (R3)*
---

---
# Step 08: *Neural Networks (R4)*
---

---
# Step 09: *Final Comparison and the Conclusion (R4)*
---